# Workshop 4 - Data Structure

## Part 2: Creating a Database from a Spreadsheet with Raw SQL

This notebook contains the second half of Workshop 4. Here, we will use Python to load data from a spreadsheet that contains chemical data (`cpd_list.xlsx`), create an SQLite database, insert the data into a table, and perform SQL queries directly using raw SQL to explore the data. 

The spreadsheet `cpd_list.xlsx` contains compounds that have been flagged by a scientist during an initial study. It includes the name of the compounds, the reason the compound was flagged, and the parent name of that compound. You will see this spreadsheet again in Workshop 6

## Step 0: Install Python Modules

Depending on your Python / Anaconda installation, you may need to install the `openpyxl` Python module. Look back at the Notebook from the first half of the workshop to figure out how to do this (specifically section 1.1)

## Step 1: Import Necessary Libraries

In [13]:
import pandas as pd
import sqlite3

## Step 2: Load the Spreadsheet Data

In [14]:
# Load the spreadsheet into a Pandas DataFrame
file_path = 'cpd_list.xlsx'
df = pd.read_excel(file_path)

# Display the first few rows of the DataFrame to verify the data
df.head()

,Compound Name,Reason,Parent Name
0,Asprin,NaN,NaN
1,Acetylsalicylic Acid,Synonyn,Asprin
2,BAY1019036,Synonyn,Asprin
3,CARBASPIRIN,Similarity,Asprin
4,DIPYROCETYL,Similarity,Asprin


### Exercise 1
- What happens if you input a number as the argument of df.head()?
- What happens if you replace df.head() with df.tail()?
- What happens if, instead of df.head(), you just type df?

## Step 3: Data Cleaning 

Depending on the quality of your data, you may need to perform some data cleaning. This might include handling missing values, renaming columns, or formatting data types.

Make sure to read the comments to see what the code does.

In [15]:
# Correct the spelling of Aspirin
df = df.replace('Asprin', 'Aspirin')

# A more complicated example - Rename columns to remove spaces, special characters, and make everything lower case
df.columns = df.columns.str.replace(' ', '_').str.lower()

# Display the cleaned DataFrame
df

,compound_name,reason,parent_name
0,Aspirin,NaN,NaN
1,Acetylsalicylic Acid,Synonyn,Aspirin
2,BAY1019036,Synonyn,Aspirin
3,CARBASPIRIN,Similarity,Aspirin
4,DIPYROCETYL,Similarity,Aspirin
...,...,...,...
85,phenylsulfanol,Not in ChEMBL,NaN
86,2-amino-2-phenylacetonitrile,Not in ChEMBL,NaN
87,callipeltin A,Not in ChEMBL,NaN
88,phenylalaninium,Not in ChEMBL,NaN


Finally, let's get rid of those NaN values where the spreadsheet was blank - this will save us some headaches later:

In [16]:
df = df.fillna('missing')
df

,compound_name,reason,parent_name
0,Aspirin,missing,missing
1,Acetylsalicylic Acid,Synonyn,Aspirin
2,BAY1019036,Synonyn,Aspirin
3,CARBASPIRIN,Similarity,Aspirin
4,DIPYROCETYL,Similarity,Aspirin
...,...,...,...
85,phenylsulfanol,Not in ChEMBL,missing
86,2-amino-2-phenylacetonitrile,Not in ChEMBL,missing
87,callipeltin A,Not in ChEMBL,missing
88,phenylalaninium,Not in ChEMBL,missing


## Step 4: Create SQLite Database and Table

Now let's starting using Python and SQL to interact with our data. You can find a nice run-through of the material we're using [at the official Python documentation](https://docs.python.org/3/library/sqlite3.html).

You can test the code below has worked by looking in the directory that this code is stored in - it should save a file called `cpd_database.db`.

First - let's look at some (not-ideal) Python for generating your table in SQL:

In [17]:
# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('cpd_database.db')

# Create a database 'cursor' - this allows us to interact with the database via SQL
cursor = conn.cursor()

# The Python interface for SQL is a little clunky - you generate one big string of SQL that is then executed.
create_table_sql = 'CREATE TABLE IF NOT EXISTS cpd_data (compound_name TEXT, reason TEXT, parent_name TEXT);'

# Execute the SQL to create the table
cursor.execute(create_table_sql)
conn.commit()

print(create_table_sql)

CREATE TABLE IF NOT EXISTS cpd_data (compound_name TEXT, reason TEXT, parent_name TEXT);


### Exercise 2
Rather than writing the SQL command out by hand, it's (arguably) better to get Python to assemble the command based on the data you give it. 

There's an example of how to do this below - compare the output of the two pieces of code:

In [18]:
# Create a table based on the DataFrame columns
columns = df.columns
column_types = ["TEXT"] * len(columns)  # Assuming all columns can be stored as TEXT, customize if needed
zipped_data = zip(columns, column_types)
table_name = 'cpd_data'

create_table_sql = f"CREATE TABLE IF NOT EXISTS {table_name} (" + ", ".join([f"{col} {col_type}" for col, col_type in zip(columns, column_types)]) + ");"
print(create_table_sql)

CREATE TABLE IF NOT EXISTS cpd_data (compound_name TEXT, reason TEXT, parent_name TEXT);


Now answer the following questions:

1. What is `zip()` doing? 

The following code might help you understand:

```python
zipped_data = zip(columns, column_types)
for n in zipped_data:
    print n
```



2. What is `f"{col} {col_type}" for col, col_type in zip(columns, column_types)` doing? 

The following code might help you understand:

```python
big_string = (f"{col} {col_type}" for col, col_type in zip(columns, column_types))
for n in big_string:
    print(n)
```

3. In the above two examples, what happens if you just use `print()`, without the `for` loop?

4. What does the `f` before each string mean? What happens if you remove it?

5. What does `join()` do in the above code?

## Step 5: Insert Data into the Table

### 5.1 - A Simple Example
First, let's just insert a single row from our DataFrame into our database:

In [19]:
row = tuple(df.iloc[1])
insert_sql = f"INSERT INTO {table_name} VALUES {row}"
cursor.execute(insert_sql)
conn.commit()

Now query the database to see if the insert worked:

In [20]:
# Example: Select the first 5 rows from the table
query = f"SELECT * FROM {table_name};"
cursor.execute(query)
rows = cursor.fetchall()

# Display the query results
for row in rows:
    print(row)

('Acetylsalicylic Acid', 'Synonyn', 'Aspirin')


### 5.2 - Inserting a Full DataFrame
Now let's use a for loop to insert each row of our DataFrame, one-by-one:

In [21]:
for n in range(0, len(df)):
    row = tuple(df.iloc[n])
    insert_sql = f"INSERT INTO {table_name} VALUES {row}"
    cursor.execute(insert_sql)
    conn.commit()

Again, let's check that works - we'll just print the first 5 results:

In [22]:
# Example: Select the first 5 rows from the table
query = f"SELECT * FROM {table_name} LIMIT 5;"
cursor.execute(query)
rows = cursor.fetchall()

# Display the query results
for row in rows:
    print(row)

('Acetylsalicylic Acid', 'Synonyn', 'Aspirin')
('Aspirin', 'missing', 'missing')
('Acetylsalicylic Acid', 'Synonyn', 'Aspirin')
('BAY1019036', 'Synonyn', 'Aspirin')
('CARBASPIRIN', 'Similarity', 'Aspirin')


## Step 6: Close the Database Connection

It's important to close the database connection when you're done:

In [23]:
conn.close()

### Think, Share, Pair Exercise
<b>If you have any time left</b>, think about the following, share with your neighbours, and then let's discuss:

- What could go wrong here if your computer broke during the execution of the command?
- How would you go about fixing / avoiding that?

### Stretch Exercise 1 - Inserting Data Pythonically
<b>You're not expected to get to this point during the workshop - but make sure you look through this in your own time at home!</b>

A more elegant way of adding data to your database is shown below - work through it line-by-line, as in Exercise 2, to see how it works. 

In [ ]:
# Insert DataFrame rows into the SQLite table using raw SQL
for index, row in df.iterrows():
    placeholders = ", ".join(["?" for _ in range(len(row))])
    insert_sql = f"INSERT INTO {table_name} VALUES ({placeholders})"
    cursor.execute(insert_sql, tuple(row))
    
conn.commit()

### Stretch Exercise 2 - Handling Missing Data

- Look up how missing data can affect SQL queries, especially in scientific datasets.
- Modify the above queries to handle missing values (e.g., using COALESCE to provide default values).
- This about how you would analyze the implications of ignoring or filling missing data in chemical experiments?

## Conclusion

In this notebook, we successfully loaded data from a spreadsheet into a Pandas DataFrame, performed optional data cleaning, and then used raw SQL to create an SQLite database and table. We also inserted the data into the table and demonstrated how to query the database directly using raw SQL.

### Homework Exercise (optional submission!)
This week's homework is a tricky one - like all homeworks, submission is completely optional.

Write a Python script that acquires data from your MKR and uses SQLite to save it into a database. If you like, submit as a .py or .ipynb file. You'll want to think about the following:

1. What tables do you need to create and what do they need to contain? Is one table enough, or can you think of more useful ways of structuring the data?
1. Beyond just data, what else could you include? In particular, what *metadata* (look the term up) could you add? (think what data about the experiment it would be useful to include! e.g., date, location, etc.)

Note that you're assessed for the way you store your data in your project, and so solutions to this exercise won't be uploaded! 




### Alternatives to Writing SQL Directly
Writing SQL directly into Python is quite painful, isn't it?

This can be massively simplified by an ORM like Django https://www.djangoproject.com/ 

Read about Django and have a think about how you would use it in this setting.